# Module B05 — Collections

## Exercise 4: Records, filtered and summarised

Every container so far has held one kind of simple thing. Real data is not
shaped like that. A student is not a name; a student is a name and a course and
a mark and a year. An order is not a total; it is a customer and a date and
several lines.

The standard shape for that in Python is a **list of dictionaries**. One
dictionary is one record. The list is all of them. Almost every file you read,
every response an API sends you, and every row set a database returns arrives in
that shape or is turned into it immediately.

This notebook takes a list of records and does the four things people do with
one: reach into it, filter it, summarise it, and print it. The failure it runs
on purpose is the one everybody meets on their first day of nesting.

| | |
|---|---|
| Time | About 50 minutes |
| You need | This notebook |
| Comes after | Exercise 3, sets and tuples |

---

## 1. One record is a dictionary

Exercise 2 used dictionaries as a lookup: many keys, all the same kind of thing,
one value each. A record uses the same container for a different job. The keys
are the **field names** and there are only a few of them, the same few in every
record.

In [ ]:
student = {"name": "ama", "course": "physics", "mark": 72}

print(student)
print(student["name"], "scored", student["mark"])

Nothing new is happening there. It is worth naming the change of intent anyway,
because it changes what good code looks like. In a phonebook you do not know the
keys while you are writing, so you use `.get` and loops. In a record you do know
them, you wrote them, and `student["mark"]` is a reasonable claim to make.

The advantage over a list like `["ama", "physics", 72]` is that `student["mark"]`
says what it is fetching. `student[2]` does not, and stops being right the day
somebody adds a field in the middle.

---

## 2. Many records are a list of dictionaries

```
students = [
    {"name": "ama",   "course": "physics", "mark": 72},     <- one record
    {"name": "kofi",  "course": "physics", "mark": 55},     <- another
    {"name": "yaa",   "course": "biology", "mark": 88},
]
│   │                                                  │
│   └── each item of the list is a whole dictionary     │
└── an ordinary list ──────────────────────────────────┘
```

The list is ordered and can be appended to. Each item is a dictionary with the
same field names.

A trailing comma after the last item is allowed and normal. It means adding the
next record touches one line rather than two.

In [ ]:
students = [
    {"name": "ama", "course": "physics", "mark": 72},
    {"name": "kofi", "course": "physics", "mark": 55},
    {"name": "yaa", "course": "biology", "mark": 88},
    {"name": "kwame", "course": "biology", "mark": 41},
]

print("records:", len(students))
print(students[0])

---

## 3. Reaching in takes two steps

`students[0]` gives you a dictionary. Asking that dictionary for a key gives you
a value. Two operations, and it helps to write them apart once before you write
them together.

In [ ]:
first = students[0]

print("type of students[0]:", type(first))
print("the record:         ", first)
print("its name field:     ", first["name"])

print()
print("both steps at once:", students[0]["name"])

`students[0]["name"]` is read left to right: take the list, take item 0 from it,
take the `"name"` key from that. Each bracket is one step inwards.

The two kinds of bracket look identical and mean different things, and that is
the whole difficulty of nesting. `[0]` is a position because a list is on its
left. `["name"]` is a key because a dictionary is on its left. What is on the
left decides what the brackets mean.

---

## 4. Asking the list for a key, run on purpose

Here is what happens when the two steps get collapsed into one.

In [ ]:
print(students["name"])

```
TypeError: list indices must be integers or slices, not str
```

**`TypeError`** means the type was wrong, and this message is unusually direct.
An **index** is what goes inside the brackets. `students` is a list, a list is
indexed by whole numbers or slices, and `"name"` is neither.

There is no sensible answer to the question, which is why this is a crash and not
a guess. `students` holds four records. There is no single name in it, and
Python will not pick one.

The fix is to say which record: `students[0]["name"]`, or a loop that visits
every record in turn, which is section 5.

The mirror image of this error is worth recognising too, because you will hit it
within the week. `students[0][0]` raises `KeyError: 0`, since `students[0]` is a
dictionary and `0` is being read as a key it has not got. Two different messages,
and each one is telling you what type it thinks it is holding.

---

## 5. A loop gives you one record per pass

Nothing new in the loop. What is new is that each pass hands you a dictionary
rather than a simple value.

In [ ]:
for student in students:
    print(student["name"].ljust(8) + student["course"].ljust(10)
          + str(student["mark"]))

`str(...)` around the mark because `+` joins text to text, which module B02
established, and `.ljust` pads on the right to a fixed width, which exercise 2 of
this module used.

The name `student` is singular and the list is plural. That is a convention worth
adopting on the day you start, because it makes `student["mark"]` read correctly
and `students["mark"]` look wrong on sight, which is the section 4 error caught
by eye instead of by traceback.

---

## 6. Filtering builds a new list of records

Filtering is the accumulator pattern from module B04 exercise 2, with whole
records as the items: start an empty list above the loop, append inside an `if`,
use it after.

In [ ]:
physicists = []
for student in students:
    if student["course"] == "physics":
        physicists.append(student)

print("matched:", len(physicists), "of", len(students))
for student in physicists:
    print("  ", student["name"], student["mark"])

`physicists` is a new list holding the records that matched. It is not a copy of
them; it is a second list pointing at the same four dictionaries, and exercise 6
is about what follows from that. For reading and reporting, which is all this
notebook does, it makes no difference.

Two conditions combine with `and` and `or`, from module B03.

In [ ]:
strong_physicists = []
for student in students:
    if student["course"] == "physics" and student["mark"] >= 60:
        strong_physicists.append(student)

print([s["name"] for s in strong_physicists] if False else strong_physicists)

That last line has a trick in it that you have not been taught, and it was left
in on purpose so that you can see it does nothing: the `if False` means the whole
first half is never used, and what actually prints is `strong_physicists`. Ignore
the shape; a later module covers it. Here is the plain version.

In [ ]:
for student in strong_physicists:
    print(student["name"], student["mark"])

---

## 7. Summarising walks the records once

A total, a count, and an average are the accumulator pattern again. Finding the
largest is the same pattern with a comparison instead of an addition: hold the
best record seen so far, and replace it when a better one turns up.

In [ ]:
total = 0
for student in students:
    total += student["mark"]

average = total / len(students)

best = students[0]
for student in students:
    if student["mark"] > best["mark"]:
        best = student

print("records: ", len(students))
print("total:   ", total)
print("average: ", round(average, 1))
print("best:    ", best["name"], "with", best["mark"])

`best = students[0]` before the loop is what makes the comparison possible.
Starting from `0` would work for marks and break for temperatures below freezing,
so starting from the first record is the version that is right for any data.

The division is unguarded. On an empty list, `len(students)` is `0` and it raises
`ZeroDivisionError`, which module B04 exercise 2 covered. `if students:` before
the division is the guard, and deciding what to report when there are no records
is a design decision rather than an accident.

---

## 8. A record with a field missing, run on purpose

Records are supposed to have the same fields. Data that came from a file, a form,
or another system is supposed to do a great many things.

In [ ]:
mixed = [
    {"name": "ama", "mark": 72},
    {"name": "kofi"},
    {"name": "yaa", "mark": 88},
]

for student in mixed:
    print(student["name"], student["mark"])

```
KeyError: 'mark'
```

**`KeyError`**, from exercise 2, and the message is the missing key. Note where
it stopped: after the first line printed. Half a report came out and then the
program died, which is the worst of both, because now somebody has to work out
whether the half that printed was written to anything.

`.get(key, default)` is the tool, and this is exactly the situation it was made
for: data you did not create, with fields that may not be there.

In [ ]:
for student in mixed:
    print(student["name"].ljust(8) + str(student.get("mark", "no mark")))

Every record produced a line, and the missing one is visible as missing rather
than as a zero. A `0` there would be a lie: it says Kofi sat the exam and scored
nothing.

When the field is a number you are about to add up, the choice matters more.
`.get("mark", 0)` puts a zero into the total. Skipping the record with
`if "mark" in student:` leaves it out of the total **and** out of the count, which
changes the average. Both are defensible and they give different answers, so it
is a decision to make on purpose.

In [ ]:
total = 0
counted = 0
for student in mixed:
    if "mark" in student:
        total += student["mark"]
        counted += 1

print("marks counted:", counted, "of", len(mixed))
print("average of the marks present:", round(total / counted, 1))

---

## 9. Sorting records by a field

`sorted(students)` does not work: it would have to compare two dictionaries and
decide which is larger, and there is no answer to that.

The way to do it with what you have is to build a list of **tuples** with the
field you are sorting by first, sort that, and read the answers off. Exercise 3
introduced tuples: a fixed group in round brackets, which unpacks into separate
names.

Sorting a list of tuples compares the first item of each, and only looks at the
second when the first two are equal.

In [ ]:
print(sorted([(72, "ama"), (55, "kofi"), (88, "yaa"), (41, "kwame")]))

So build those pairs from the records, sort, and unpack them back out.

In [ ]:
pairs = []
for student in students:
    pairs.append((student["mark"], student["name"]))

pairs.sort(reverse=True)

for mark, name in pairs:
    print(str(mark).rjust(3), name)

Note the double brackets in `pairs.append((student["mark"], student["name"]))`.
The outer pair belongs to `append`, the inner pair makes the tuple. Writing one
pair appends two items and raises `TypeError: append() takes exactly one
argument`, which is a confusing message for a missing bracket.

There is a shorter way, `sorted(students, key=...)`, which the intermediate level
teaches. The tuple version is worth writing once anyway, because it shows what
sorting by a field actually means.

---

## 10. A report

Everything together: a filter, a summary, and columns that line up.

In [ ]:
print("Physics results")
print("-" * 28)

total = 0
counted = 0
for student in students:
    if student["course"] != "physics":
        continue
    print(student["name"].ljust(10) + str(student["mark"]).rjust(4))
    total += student["mark"]
    counted += 1

print("-" * 28)
if counted:
    print("average".ljust(10) + str(round(total / counted, 1)).rjust(4))
else:
    print("no physics students")

`continue` from module B04 exercise 3 skips the records that do not belong,
which saves indenting the whole body one level deeper. The guard on `counted`
means the report survives a filter that matches nothing.

---

## 11. The other shape: a dictionary of records

A list of records is not the only shape. When every record has an identifier and
you look them up by it, a dictionary of records is the better fit.

In [ ]:
students_by_id = {
    "s101": {"name": "ama", "course": "physics", "mark": 72},
    "s102": {"name": "kofi", "course": "physics", "mark": 55},
    "s103": {"name": "yaa", "course": "biology", "mark": 88},
}

print(students_by_id["s102"]["name"])

for student_id, student in students_by_id.items():
    print(student_id, student["name"])

Finding `s102` in that dictionary is one lookup. Finding it in a list of records
means walking the list until you hit it, which is the cost exercise 3 measured.

The trade is what you lose. A dictionary of records cannot hold two records with
the same id, which is a feature when ids are unique and a data-loss bug when they
are not. And the list is the shape data arrives in, so somebody has to do the
conversion.

Which shape to hold is task 5.

---

# Your turn

**Do not delete the `# ANSWER n` marker lines.** The self-check uses them.

### Task 1

Fill in every prediction **before you run the cell**. For the types, write the
word `list`, `dict`, `str`, or `int`.

In [ ]:
# ANSWER 1
# type of orders          -> ___
# type of orders[1]       -> ___
# type of orders[1]["id"] -> ___
# orders[1]["total"]      -> ___
# orders[-1]["customer"]  -> ___
# len(orders)             -> ___

orders = [
    {"id": "a1", "customer": "ama", "total": 40},
    {"id": "a2", "customer": "kofi", "total": 125},
    {"id": "a3", "customer": "yaa", "total": 12},
]

print("type of orders          ->", type(orders))
print("type of orders[1]       ->", type(orders[1]))
print("type of orders[1]['id'] ->", type(orders[1]["id"]))
print("orders[1]['total']      ->", orders[1]["total"])
print("orders[-1]['customer']  ->", orders[-1]["customer"])
print("len(orders)             ->", len(orders))

### Task 2

Filter. Build a new list holding only the orders over 20, then report how many
of the orders that was.

In [ ]:
# ANSWER 2
orders = [
    {"id": "a1", "customer": "ama", "total": 40},
    {"id": "a2", "customer": "kofi", "total": 125},
    {"id": "a3", "customer": "yaa", "total": 12},
    {"id": "a4", "customer": "ama", "total": 8},
]

large = ___

for order in orders:
    if ___ > 20:
        large.___(order)

print("large orders:", len(large), "of", len(orders))
for order in large:
    print("  ", order["id"], order["customer"], order["total"])

### Task 3

Summarise. The total of every order, the average, and the single largest order.

The largest must be found by walking the records, not by sorting.

In [ ]:
# ANSWER 3
total = 0
for order in orders:
    total += ___

biggest = orders[___]
for order in orders:
    if order["total"] > ___["total"]:
        biggest = ___

print("orders: ", len(orders))
print("total:  ", total)
print("average:", round(total / ___, 2))
print("biggest:", biggest["id"], "at", biggest["total"])

### Task 4

These records came from another system and two of them are incomplete. Print a
line for every one of them, with a stand-in where a field is missing, and then a
total that counts only the orders that have one.

Nothing in this cell may raise.

In [ ]:
# ANSWER 4
# This template will not run until you fill the blanks: some of them
# stand where an operator or a keyword goes, not where a value goes.
messy = [
    {"id": "b1", "customer": "ama", "total": 40},
    {"id": "b2", "customer": "kofi"},
    {"id": "b3", "total": 12},
    {"id": "b4", "customer": "yaa", "total": 30},
]

total = 0
counted = 0

for order in messy:
    customer = order.___("customer", ___)
    amount = order.___("total", ___)
    print(order["id"].ljust(4) + str(customer).ljust(8) + str(amount))

    if "total" ___ order:
        total += order["total"]
        counted += 1

print("counted", counted, "of", len(messy), "orders, totalling", total)

### Task 5

You are holding 50,000 orders in memory. The program does two things with them
all day: it looks up single orders by their id, and it prints them in the order
they arrived.

```python
# shape one: a list of records
orders = [{"id": "a1", ...}, {"id": "a2", ...}]

# shape two: a dictionary of records, keyed by id
orders = {"a1": {...}, "a2": {...}}
```

Say which you would hold, why, and what fact about the ids would change your
answer. Then say what shape two makes harder.

There is no correct answer. The reasoning is the exercise.

In [ ]:
# ANSWER 5
which_shape_and_why = "___"
what_about_the_ids_would_change_it = "___"
what_shape_two_makes_harder = "___"

---

## Self-check

You do not need to understand this cell. It is machinery, not material.

In [ ]:
def _answer(marker):
    """Find the most recent cell you ran that contains the given marker."""
    try:
        matches = [c for c in _ih if marker in c and "def _answer" not in c]
    except NameError:
        print("Run this in Jupyter or VS Code so the self-check can see your cells.")
        return ""
    return matches[-1] if matches else ""


def check(passed, message):
    print(("PASS  " if passed else "FAIL  ") + message)
    return bool(passed)

NEXT = "Move on to exercise 5, the written worksheet on choosing a container."

a1, a2, a3, a4, a5 = (_answer("# ANSWER 1"), _answer("# ANSWER 2"),
                      _answer("# ANSWER 3"), _answer("# ANSWER 4"),
                      _answer("# ANSWER 5"))

results = [
    check("type of orders          -> ___" not in a1 and a1.count("___") == 0,
          "Task 1: all six predictions written before running"),
    check(a2.count("___") == 0 and "large=[]" in a2.replace(" ", ""),
          "Task 2: the new list starts empty above the loop"),
    check('order["total"]' in a2 and ".append(order)" in a2.replace(" ", ""),
          "Task 2: the whole record is appended, chosen on its total field"),
    check(a3.count("___") == 0 and 'order["total"]' in a3,
          "Task 3: the total accumulates the mark field of each record"),
    check("biggest = order" in a3 and "orders[0]" in a3,
          "Task 3: the largest starts at the first record and is replaced as you go"),
    check(a4.count("___") == 0 and a4.count(".get(") >= 2,
          "Task 4: both fields read with get and a default"),
    check('"total" in order' in a4,
          "Task 4: the total only counts records that have the field"),
    check(a5.count("___") == 0 and "what_shape_two_makes_harder" in a5,
          "Task 5: a shape chosen, a condition that reverses it, and a cost named"),
]

print()
failed = results.count(False)
print("%d of %d checks failing. Keep going." % (failed, len(results)) if failed
      else "All %d checks passing. %s" % (len(results), NEXT))

---

## What you learned

- One record is a dictionary whose keys are field names, and `record["mark"]`
  says what it is fetching in a way `record[2]` cannot.
- Many records are a list of dictionaries, which is the shape most real data
  arrives in.
- Reaching in is two steps. `students[0]["name"]` is a position then a key, and
  what is on the left of the brackets decides what they mean.
- `students["name"]` raises `TypeError: list indices must be integers`, and
  `students[0][0]` raises `KeyError: 0`. Each message names the type it is
  holding.
- A loop over records hands you a whole dictionary per pass, and singular and
  plural names make the mistake visible by eye.
- Filtering is the accumulator pattern with records as the items.
- Summarising walks the records once. The largest starts at the first record
  rather than at zero.
- A missing field raises `KeyError` part way through a report. `.get(key,
  default)` prints every record, and whether to default a number or skip the
  record changes the answer.
- Sorting by a field means building tuples with that field first, sorting those,
  and unpacking them back out.
- A dictionary of records keyed by id makes lookup immediate and makes duplicate
  ids impossible, which is a feature or a data loss depending on your data.

## Before you move on

- [ ] You ran `students["name"]` and can say why there is no sensible answer to
      it.
- [ ] You can write the two steps of `students[0]["name"]` apart and say what
      each one produces.
- [ ] You filtered a list of records into a new list without a `KeyError`.
- [ ] You have an opinion about defaulting a missing number to zero.

**Next:** exercise 5, a written worksheet with no code in it, where you choose a
container for a series of situations and defend the choice. Then exercise 6,
which is about what `b = a` really does.